# 01 — Generate synthetic shots

Lays down `shots_raw` — ~50k synthetic NHL shots with realistic distributions and a
**hidden ground-truth `P(goal)`** function so the models in `03` actually have
signal to learn.

Shot location uses the standard 200×85ft rink coordinates with the offensive
goal at `(89, 0)`. Distributions are picked to roughly match league-wide
shooting %, not any specific team. **The numbers are not real hockey** — but the
shape of the data is identical to what an event-tracking provider would give you,
so the downstream MLflow story is real.

In [1]:
import os, math
import numpy as np
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "hockey_xg_mlflow")
TABLE      = f"{UC_CATALOG}.{UC_SCHEMA}.shots_raw"

rng = np.random.default_rng(seed=42)
N_SHOTS = 50_000
N_TEAMS = 32
N_GAMES = 820     # ~ full NHL regular season
PLAYERS_PER_TEAM = 25
print(f"Generating {N_SHOTS:,} shots across {N_GAMES} games / {N_TEAMS} teams → {TABLE}")

/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:141: UserWarning: Could not enable SetAllowOversizeProtos, please check the protobuf version.
  warnings.warn("Could not enable SetAllowOversizeProtos, please check the protobuf version.")


Generating 50,000 shots across 820 games / 32 teams → alexander_booth.hockey_xg_mlflow.shots_raw


## Rink geometry + distributions

All shots are in the offensive zone (`x ∈ [25, 89]`, `y ∈ [-42.5, 42.5]`), goal
at `(89, 0)`. Distance and angle are derived from `(x, y)`.

We sample `x` close to the goal (most shots come from inside ~40ft) and `y`
around the slot. Then we layer in categorical features.

In [2]:
# Shot location — biased toward the slot / high-danger area
x = 89 - np.clip(rng.gamma(shape=2.0, scale=12.0, size=N_SHOTS), 1, 64)
y = rng.normal(loc=0.0, scale=15.0, size=N_SHOTS)
y = np.clip(y, -42.0, 42.0)

# Derived geometry
dx = 89.0 - x
dy = y
distance_ft = np.sqrt(dx**2 + dy**2)
# Angle to the center of the goal mouth, measured from the goal line normal
angle_deg = np.degrees(np.arctan2(np.abs(dy), np.maximum(dx, 0.5)))

print(f"Distance: mean={distance_ft.mean():.1f}ft, p50={np.median(distance_ft):.1f}, max={distance_ft.max():.1f}")
print(f"Angle:    mean={angle_deg.mean():.1f}°, p50={np.median(angle_deg):.1f}, max={angle_deg.max():.1f}")

Distance: mean=28.3ft, p50=25.8, max=76.6
Angle:    mean=30.6°, p50=26.0, max=88.6


In [3]:
# Categorical features
shot_types   = np.array(["wrist", "snap", "slap", "backhand", "tip", "wrap"])
shot_p       = np.array([0.42,    0.20,   0.15,   0.10,       0.10,  0.03])
shot_type    = rng.choice(shot_types, size=N_SHOTS, p=shot_p)

strength_states = np.array(["5v5", "5v4_pp", "4v5_pk", "4v4", "3v3_ot", "6v5_en", "5v6"])
strength_p      = np.array([0.78,   0.10,    0.03,     0.04,   0.01,     0.02,     0.02])
strength_state  = rng.choice(strength_states, size=N_SHOTS, p=strength_p)

period        = rng.choice([1, 2, 3, 4], size=N_SHOTS, p=[0.32, 0.33, 0.32, 0.03])
rebound       = rng.random(N_SHOTS) < 0.11
rush          = rng.random(N_SHOTS) < 0.16
# Time since previous event (seconds) — rebounds are very fast follow-ups
prev_event_sec = np.where(
    rebound,
    rng.uniform(0.5, 3.0, size=N_SHOTS),
    rng.exponential(scale=18.0, size=N_SHOTS),
)

In [4]:
# Game / team / player identifiers
game_id    = rng.integers(low=2024020001, high=2024020001 + N_GAMES, size=N_SHOTS)
team_id    = rng.integers(low=1, high=N_TEAMS + 1, size=N_SHOTS)
opp_offset = rng.integers(low=1, high=N_TEAMS, size=N_SHOTS)
opp_team_id = ((team_id - 1 + opp_offset) % N_TEAMS) + 1
player_id  = (team_id - 1) * PLAYERS_PER_TEAM + rng.integers(low=1, high=PLAYERS_PER_TEAM + 1, size=N_SHOTS)

# Time within period in seconds, 0..1200 (20 minutes)
time_in_period_sec = rng.integers(low=0, high=1200, size=N_SHOTS)

## Hidden ground-truth `P(goal)`

We construct a logit that any reasonable model should be able to recover from
the features. This is deliberately not linear in everything — `XGBoost` will
pick up the interactions that the logistic regression can't.

In [5]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

shot_type_eff = pd.Series(shot_type).map({
    "wrist":    0.00,
    "snap":     0.10,
    "slap":    -0.10,
    "backhand":-0.25,
    "tip":      0.55,
    "wrap":    -0.05,
}).to_numpy()

strength_eff = pd.Series(strength_state).map({
    "5v5":     0.00,
    "5v4_pp":  0.35,
    "4v5_pk": -0.30,
    "4v4":     0.10,
    "3v3_ot":  0.25,
    "6v5_en":  2.50,   # net empty → huge bump
    "5v6":    -0.40,
}).to_numpy()

logit = (
    -0.70                                                # base rate (calibrated to ~9% league avg)
    + -0.055 * distance_ft                               # closer = better
    + -0.022 * angle_deg                                 # straighter = better
    + 0.0006 * (distance_ft ** 2) / 10                   # mild nonlinearity
    + shot_type_eff
    + strength_eff
    + 0.75 * rebound
    + 0.40 * rush
    + 0.10 * (period == 3).astype(float)                 # 3rd-period push
    + rng.normal(0, 0.20, size=N_SHOTS)                  # residual noise
)

p_goal = sigmoid(logit)
goal = (rng.random(N_SHOTS) < p_goal).astype(int)
print(f"Overall goal rate: {goal.mean()*100:.2f}%   (NHL is ~9-10%)")

Overall goal rate: 9.42%   (NHL is ~9-10%)


In [6]:
pdf = pd.DataFrame({
    "shot_id":            np.arange(1, N_SHOTS + 1),
    "game_id":            game_id,
    "period":             period.astype(int),
    "time_in_period_sec": time_in_period_sec.astype(int),
    "team_id":            team_id.astype(int),
    "opp_team_id":        opp_team_id.astype(int),
    "player_id":          player_id.astype(int),
    "x":                  x.round(2),
    "y":                  y.round(2),
    "distance_ft":        distance_ft.round(2),
    "angle_deg":          angle_deg.round(2),
    "shot_type":          shot_type,
    "strength_state":     strength_state,
    "rebound":            rebound,
    "rush":               rush,
    "prev_event_sec":     prev_event_sec.round(2),
    "goal":               goal.astype(int),
})
pdf.head()

,shot_id,game_id,period,time_in_period_sec,team_id,opp_team_id,player_id,x,y,distance_ft,angle_deg,shot_type,strength_state,rebound,rush,prev_event_sec,goal
0,1,2024020124,2,644,25,32,612,63.90,22.41,33.65,41.75,wrist,5v5,False,False,0.50,0
1,2,2024020197,1,932,19,25,474,54.98,-1.20,34.05,2.02,tip,5v5,False,False,0.65,0
2,3,2024020372,1,195,5,20,123,66.95,23.23,32.03,46.50,wrist,5v5,False,False,20.52,0
3,4,2024020583,1,645,29,26,723,69.26,-3.98,20.14,11.39,wrist,5v5,False,False,40.08,0
4,5,2024020725,2,133,30,24,739,52.05,-26.72,45.60,35.87,backhand,5v5,False,False,6.37,0


In [7]:
# Quick sanity check on the conditional distributions
print("Goal rate by shot_type:")
print(pdf.groupby("shot_type")["goal"].agg(["mean", "count"]).sort_values("mean", ascending=False))
print("\nGoal rate by strength_state:")
print(pdf.groupby("strength_state")["goal"].agg(["mean", "count"]).sort_values("mean", ascending=False))
print("\nGoal rate by rebound flag:")
print(pdf.groupby("rebound")["goal"].agg(["mean", "count"]))

Goal rate by shot_type:
               mean  count
shot_type                 
tip        0.134908   4996
snap       0.098977   9972
wrist      0.092164  21082
slap       0.086471   7436
wrap       0.079845   1553
backhand   0.067930   4961

Goal rate by strength_state:
                    mean  count
strength_state                 
6v5_en          0.456842    950
5v4_pp          0.109970   5065
3v3_ot          0.107280    522
5v5             0.085887  38900
4v4             0.079227   2070
4v5_pk          0.063514   1480
5v6             0.061204   1013

Goal rate by rebound flag:
             mean  count
rebound                 
False    0.087069  44436
True     0.150791   5564


## Land it in Unity Catalog

Write `shots_raw` as a Delta table. We use `mode("overwrite")` so re-running
this notebook is safe — the table just gets replaced with a fresh sample (same
seed = same data).

In [8]:
sdf = spark.createDataFrame(pdf)
(sdf.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE))

spark.sql(f"COMMENT ON TABLE {TABLE} IS 'Synthetic NHL-style shot events. ~50k rows, 1 simulated season. Generated by 01_generate_synthetic_shots.ipynb — DO NOT use for real analysis.'")
print(f"Wrote {sdf.count():,} rows to {TABLE}")
spark.sql(f"SELECT * FROM {TABLE} LIMIT 5").show(truncate=False)

Wrote 50,000 rows to alexander_booth.hockey_xg_mlflow.shots_raw


+-------+----------+------+------------------+-------+-----------+---------+-----+-----+-----------+---------+---------+--------------+-------+-----+--------------+----+
|shot_id|game_id   |period|time_in_period_sec|team_id|opp_team_id|player_id|x    |y    |distance_ft|angle_deg|shot_type|strength_state|rebound|rush |prev_event_sec|goal|
+-------+----------+------+------------------+-------+-----------+---------+-----+-----+-----------+---------+---------+--------------+-------+-----+--------------+----+
|6251   |2024020586|2     |257               |9      |11         |202      |82.44|29.11|29.84      |77.29    |snap     |5v5           |false  |true |13.63         |0   |
|6252   |2024020199|1     |813               |23     |1          |556      |61.96|-3.2 |27.23      |6.76     |wrist    |5v4_pp        |false  |false|20.17         |0   |
|6253   |2024020397|1     |41                |26     |2          |638      |72.92|35.71|39.16      |65.76    |snap     |5v5           |false  |false|5